# README 原生功能展示

使用内置现金贷数据：14 个数值字段用于分析，按授信月份及 `GROUP` 客群划分比较风险表现。

标签展示 MOB1 / MOB6 分别 > 0、> 30 四种口径；MOB1 为短期欺诈风险代理，MOB6 为长期信用风险。建模主标签为 **MOB6 > 30**，按客户隔离训练和测试集。正式分析仍需核实表现期成熟度；逾期表现不能直接等同于确认欺诈。

运行结果保存在系统临时目录的 `hscredit-readme-render/deliverables/` 下，可查看分箱图、策略与模型工作簿。

从仓库根目录或 examples 运行。安装 `hscredit[boost,tune,pmml]`，并为 PMML 导出准备 Java 环境。

## 运行准备

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hscredit").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import json
import tempfile
import hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text
import hscredit
from hscredit import hscredit_dataset, OptimalBinning
from hscredit.utils.fonts import get_bundled_font_path
from hscredit.core import viz
from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display, Image
out = Path(tempfile.gettempdir()) / "hscredit-readme-render" / "deliverables"
out.mkdir(parents=True, exist_ok=True)
work = Path(tempfile.gettempdir()) / "hscredit-readme-render"
table_dir = work / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
font_path = get_bundled_font_path()
hscredit.init_setting(font_path=str(font_path), seed=42)
font_name = font_manager.FontProperties(fname=str(font_path)).get_name()
# SHAP 中的数学标记也沿用当前包字体。
plt.rcParams.update({"mathtext.fontset": "custom", "mathtext.rm": font_name,
                    "mathtext.it": font_name, "mathtext.bf": font_name})
assert plt.rcParams["font.family"] == [font_name]
font_audit = {}

def show_table(table, name):
    # 原始结果直接 show，不重新拼装展示指标。
    table.show().export_html(str(table_dir / f"{name}.html"))

def show_plot(name, function, *args, **kwargs):
    result = function(*args, save=str(out / f"{name}.png"), **kwargs)
    figures = [plt.figure(number) for number in plt.get_fignums()]
    if hasattr(result, "findobj") and hasattr(result, "savefig") and result not in figures:
        figures.append(result)
    texts = [text for fig in figures for text in fig.findobj(Text) if text.get_text()]
    families = sorted({text.get_fontproperties().get_name() for text in texts})
    assert texts and families == [font_name], (name, families)
    font_audit[name] = {"字体": font_name, "文本对象数": len(texts), "字体文件SHA256": hashlib.sha256(font_path.read_bytes()).hexdigest()}
    (work / "font-audit.json").write_text(json.dumps(font_audit, ensure_ascii=False, indent=2), encoding="utf-8")
    plt.close("all")
    display(Image(filename=str(out / f"{name}.png")))

## 内置数据与客户隔离

In [ ]:
# 原始字段保持不变；只给分析副本设置中文名称。
field_map = {
    "DK_X1995_2": "非银机构借款总金额",
    "BASE_X187": "最近12个月多头查询次数",
    "DK_X44": "个人消费贷借款总金额",
    "DK_X1223": "个人消费贷款账户数",
    "DK_X1689": "现行贷款账户数",
    "DK_X1925": "个人经营性贷款账户数",
    "VAR0252": "公积金记录数",
    "DK_X2812": "现用贷款账户数量占贷款账户数量比率",
    "DK_X597": "现用消费贷本金余额求和占现用消费贷本金求和比率",
    "JH_X10": "杠杆率",
    "DJK_X1435": "现行贷记卡已用额度与授信额度比率_最大值",
    "DK_X2677": "贷款本月应还金额",
    "DK_X515": "现行房贷本月应还款_求和",
    "DK_X1948": "现用贷款本月应还款最大",
}
data = hscredit_dataset(columns=["用户编码", "授信时间", "loan_amt", "MOB1", "MOB6", "GROUP", *field_map])
df = data.rename(columns=field_map).copy()
df["授信时间"] = pd.to_datetime(df["授信时间"])
df["授信月份"] = df["授信时间"].dt.to_period("M").astype(str)
for overdue in ["MOB1", "MOB6"]:
    df[overdue] = df[overdue].where(df[overdue].notna() & df[overdue].ge(0))
# 标签各自使用有效表现，不能把缺失值当成好样本。
label_summary = []
for overdue, risk in [("MOB1", "短期欺诈风险代理"), ("MOB6", "长期信用风险")]:
    for threshold in [0, 30]:
        label = df[overdue].gt(threshold).where(df[overdue].notna())
        label_summary.append({"风险口径": risk, "标签": f"{overdue} > {threshold}",
                              "有效样本数": label.notna().sum(), "坏样本数": label.sum(), "坏样本率": label.mean()})
show_table(pd.DataFrame(label_summary), "label-summary")
df["坏样本"] = df["MOB6"].gt(30).where(df["MOB6"].notna()).astype("Int64")
analysis_features = list(field_map.values())
# 全部 14 个数值字段用于摘要与筛选；6 个字段用于模型与评分卡展示。
features = ["非银机构借款总金额", "最近12个月多头查询次数", "个人消费贷款账户数", "杠杆率", "贷款本月应还金额", "现行贷款账户数"]
model_data = df.loc[df["坏样本"].notna()].copy()
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(model_data, groups=model_data["用户编码"]))
train, test = model_data.iloc[train_idx].copy(), model_data.iloc[test_idx].copy()
X_train, X_test = train[features], test[features]
y_train, y_test = train["坏样本"].astype(int), test["坏样本"].astype(int)
fill_values = X_train.median()
X_num, X_test_num = X_train.fillna(fill_values), X_test.fillna(fill_values)
X_all = train[analysis_features].fillna(train[analysis_features].median())
assert set(train["用户编码"]).isdisjoint(set(test["用户编码"]))
show_table(df[analysis_features].summary(), "dataset-summary")

## 分箱统计与摘要

In [ ]:
from hscredit.report import feature_bin_stats, feature_binning_summary, feature_group_binning_summary, feature_bin_stats_2d
bins = feature_bin_stats(df, feature="最近12个月多头查询次数", target="坏样本", method="quantile", max_n_bins=4, margins=True, n_jobs=1)
show_table(bins, "binning")
multi_bins = feature_bin_stats(df, feature="最近12个月多头查询次数", overdue=["MOB1", "MOB6"], dpds=[0, 30], overdue_operator=">", method="quantile", max_n_bins=4, margins=True, n_jobs=1)
show_table(multi_bins, "binning-multi")
method_tables, method_summary = feature_binning_summary(df, feature="最近12个月多头查询次数", target="坏样本", methods=["quantile", "tree", "chi"], max_n_bins=4, margins=True, n_jobs=1)
show_table(method_summary, "binning-methods")
group_tables, group_summary = feature_group_binning_summary(df, feature="最近12个月多头查询次数", target="坏样本", methods="quantile", date="授信时间", freq="M", max_n_bins=4, margins=True, n_jobs=1)
show_table(group_summary, "binning-groups")
bins_2d = feature_bin_stats_2d(df, features=["最近12个月多头查询次数", "非银机构借款总金额"], target="坏样本", max_n_bins=3, margins=True, n_jobs=1)
show_table(bins_2d, "binning-2d")
cohort_tables, cohort_summary = feature_group_binning_summary(
    df, feature="最近12个月多头查询次数", methods="quantile", group="GROUP", dropna=False,
    overdue=["MOB1", "MOB6"], dpds=[0, 30], max_n_bins=4, margins=True, n_jobs=1,
)
show_table(cohort_summary, "binning-cohort")

## 原生分箱图

In [ ]:
show_plot("binning-plot", viz.bin_plot, bins, title="多头查询次数分箱 · MOB6 > 30", figsize=(12, 6), dpi=130)
show_plot("binning-multi-plot", viz.bin_overdues_plot, bin_table=multi_bins, title="短期欺诈风险代理与长期信用风险", dpi=120, n_jobs=1)
show_plot("binning-trend", viz.bin_trend_plot, df, feature="最近12个月多头查询次数", target="坏样本", date="授信时间", freq="M", max_n_bins=4, n_jobs=1, dpi=120)
show_plot("binning-2d-plot", viz.bin_2d_plot, df, features=["最近12个月多头查询次数", "非银机构借款总金额"], target="坏样本", max_n_bins=3, dpi=120)
show_plot("binning-cohort-plot", viz.bin_trend_plot, df, feature="最近12个月多头查询次数", target="坏样本", group="GROUP", max_n_bins=4, n_jobs=1, dpi=150)


## 编码器四宫格

In [ ]:
from hscredit.core.encoders import WOEEncoder, TargetEncoder, CatBoostEncoder
from sklearn.pipeline import Pipeline
binner = OptimalBinning(method="quantile", max_n_bins=4, n_jobs=1).fit(X_train, y_train)
woe = WOEEncoder(n_jobs=1).fit(binner.transform(X_train, metric="indices"), y_train)
show_table(woe.transform(binner.transform(X_test, metric="indices")).head(6), "encoder-woe")
category_train = train[["GROUP"]]
category_test = test[["GROUP"]]
target_encoder = TargetEncoder(training_mode="oof", cv=3, random_state=42, n_jobs=1)
# fit_transform 返回训练用 OOF 编码；transform 使用完整训练集学到的映射。
train_target = target_encoder.fit_transform(category_train, y_train)
show_table(target_encoder.transform(category_test).head(6), "encoder-target")
cat_encoder = CatBoostEncoder(random_state=42, n_jobs=1)
train_ordered = cat_encoder.fit_transform(category_train, y_train)
show_table(cat_encoder.transform(category_test).head(6), "encoder-catboost")
encoding_pipeline = Pipeline([("分箱", OptimalBinning(method="quantile", max_n_bins=4, n_jobs=1)), ("WOE", WOEEncoder(n_jobs=1))])
encoding_pipeline.fit(X_train, y_train)
show_table(encoding_pipeline.transform(X_test).head(6), "encoder-pipeline")

## 筛选器四宫格与 Pipeline

In [ ]:
from hscredit.core.selectors import LiftSelector, NullImportanceSelector, RFESelector, ScorecardFeatureSelection, NullSelector, CorrSelector, selection_report
from hscredit.core.selectors import collect_selection_report
from sklearn.ensemble import RandomForestClassifier
selectors = {
    "selector-lift": LiftSelector(threshold=0.1, ratio=0.2, n_jobs=1),
    "selector-null-importance": NullImportanceSelector(RandomForestClassifier(n_estimators=20, max_depth=3, random_state=42, n_jobs=1), cv=2, n_runs=2, n_jobs=1),
    "selector-rfe": RFESelector(RandomForestClassifier(n_estimators=20, max_depth=3, random_state=42, n_jobs=1), n_features_to_select=6, n_jobs=1),
    "selector-scorecard": ScorecardFeatureSelection(iv_threshold=0.01, n_jobs=1),
}
for name, selector in selectors.items():
    selector.fit(X_all, y_train)
    summary, details = selection_report(selector)
    trace = collect_selection_report(selector).get_feature_trace()
    show_table(trace, name + "-trace")
    # 首批记录是各特征的最终决策；组合器的分阶段轨迹另存完整 HTML。
    show_table(trace.head(len(analysis_features)), name)
    show_table(details, name + "-details")
selection_pipeline = Pipeline([("缺失率", NullSelector(threshold=0.95, n_jobs=1)), ("相关性", CorrSelector(threshold=0.8, bin_params={"method": "best_iv", "max_n_bins": 5}, n_jobs=1))])
selection_pipeline.fit(X_all, y_train)
selection_summary, selection_details = selection_report(selection_pipeline)
show_table(selection_summary, "selection-summary")
show_table(selection_details, "selection-details")

## LR 与评分卡

In [ ]:
from hscredit import LogisticRegression, ScoreCard
X_woe = woe.transform(binner.transform(X_train, metric="indices"))
lr = LogisticRegression(max_iter=500).fit(X_woe, y_train)
show_table(lr.summary(), "lr-summary")
card = ScoreCard(binner=binner, encoder=woe, lr_model=lr, base_score=650, base_odds=35, pdo=50)
card.fit(X_train, y_train, input_type="raw")
show_table(card.scorecard_points(), "scorecard-points")
show_table(card.scorecard_scale(), "scorecard-scale")
# 从训练样本的好坏计数独立核对 WOE，并逐笔用公开分值表还原测试集评分。
points = card.scorecard_points(decimal=8)
ids = binner.transform(X_test, metric="indices")
reconstructed = np.full(len(X_test), points.iloc[0]["对应分数"])
for j, feature in enumerate(features):
    stats = binner.bin_tables_[feature].reset_index(drop=True)
    rows = points[points["变量名称"] == feature].reset_index(drop=True)
    raw_woe = np.log(((stats["坏样本数"] + 1) / (y_train.sum() + 2)) / ((stats["好样本数"] + 1) / ((1 - y_train).sum() + 2))).clip(-5, 5)
    np.testing.assert_allclose(rows["WOE值"], raw_woe * card.lr_model_.woe_coef_signs_[j], atol=0.000051)
    reconstructed += ids[feature].map(dict(zip(stats["分箱"], rows["对应分数"]))).to_numpy()
assert np.ptp(reconstructed) > 1
np.testing.assert_allclose(card.predict(X_test, input_type="raw"), reconstructed, atol=0.0051)
np.testing.assert_allclose(card.predict_proba(X_test, input_type="raw"), lr.predict_proba(woe.transform(ids)))


## XGBoost、调参与 SHAP

In [ ]:
from hscredit import XGBoost, AutoTuner
from hscredit.core.models.losses import KSMetric
model = XGBoost(n_estimators=40, max_depth=3, n_jobs=1, random_state=42, eval_metric=KSMetric())
model.fit(X_num, y_train)
probability = model.predict_proba(X_test_num)[:, 1]
show_plot("model-importance", viz.plot_model_feature_importance, model, X_test_num.iloc[:150], y_test.iloc[:150], background_size=30, right_top_n=4, show=False, dpi=120)
show_plot("model-sample-shap", viz.plot_model_sample_shap, model, X_test_num.iloc[[0]], X_num.iloc[:40], background_size=30, figsize=(18, 12), show=False, dpi=120)
# 两种调参入口均运行小规模搜索，演示结果不代表最佳模型。
tuner = AutoTuner.create("xgboost", metric="auc", cv=2, random_state=42, n_jobs=1, search_space={"max_depth": [2, 3]}, fixed_params={"n_estimators": 20, "n_jobs": 1})
tuner.fit(X_num, y_train, n_trials=2, show_progress_bar=False)
best_auto = tuner.get_best_model()
best = model.tune(X_num, y_train, search_space={"max_depth": [2, 3], "learning_rate": [0.05, 0.1]}, metric="auc", cv=2, n_trials=2, n_jobs=1, show_progress_bar=False)
# 多目标搜索：提高 AUC，同时缩小训练与验证 KS 的差异。
storage = f"sqlite:///{(work / 'readme-tuning.db').as_posix()}"
study_key = hashlib.sha256(pd.util.hash_pandas_object(X_num.join(y_train.rename("标签")), index=True).values.tobytes()).hexdigest()[:10]
pareto_tuner = AutoTuner.create(
    "xgboost", metric=["auc", "ks_diff"], cv=2, random_state=42, n_jobs=1,
    search_space={"max_depth": [2, 3, 4], "learning_rate": [0.03, 0.1, 0.2]},
    fixed_params={"n_estimators": 30, "n_jobs": 1},
    storage=storage, study_name=f"XGBoost多目标搜索 · {len(y_train)} · {study_key}", load_if_exists=True,
)
pareto_tuner.fit(X_num, y_train, n_trials=8, show_progress_bar=False)
pareto_figure = pareto_tuner.plot_pareto_front()
pareto_figure.update_layout(width=1100, height=650)
pareto_figure.write_html(str(work / "pareto-front.html"), include_plotlyjs=True)
pareto_figure.show()


## 规则与规则集

In [ ]:
from hscredit import Rule, RuleSet, RulesClassifier
from hscredit.report import ruleset_analysis
rule1 = Rule("最近12个月多头查询次数 > 60", name="多头查询超过60次")
rule2 = Rule("贷款本月应还金额 > 20000", name="月应还额超过2万")
show_table(rule1.report(test, margins=True, overdue=["MOB1", "MOB6"], dpds=[0, 30], amount="loan_amt"), "rule-report")
rule_set = RuleSet(rules=[rule1, rule2], logic="or", n_jobs=1)
rule_model = RulesClassifier(rules=rule_set, n_jobs=1).fit(X_num, y_train)
show_table(ruleset_analysis(test, amount="loan_amt", rules=[rule1, rule2], overdue=["MOB1", "MOB6"], dpds=[0, 30], n_jobs=1), "ruleset-report")

## EDA 与六种常用图

In [ ]:
show_table(df[analysis_features + ["坏样本"]].summary(y="坏样本"), "eda-summary")
show_plot("eda-ks", viz.ks_plot, probability, y_test, title="测试集 KS / ROC", figsize=(12, 5), dpi=120)
show_plot("eda-corr", viz.corr_plot, X_all, figsize=(18, 15), fontsize=10, max_len=16, dpi=120)
show_plot("eda-hist", viz.hist_plot, probability, y_test, title="测试集预测概率", figsize=(8, 5), dpi=120)
show_plot("eda-confusion", viz.confusion_matrix_plot, y_test, probability >= 0.5, figsize=(8, 5), dpi=120)
show_plot("eda-psi", viz.psi_plot, X_train["最近12个月多头查询次数"], X_test["最近12个月多头查询次数"], labels=["训练集", "测试集"], figsize=(12, 6), dpi=150)
show_plot("eda-trend", viz.bad_rate_trend_plot, df, date="授信时间", overdue=["MOB1", "MOB6"], dpds=[0, 30], overdue_operator=">", figsize=(10, 5), dpi=120)

## 多图 Excel 与原生透视表

In [ ]:
from hscredit.excel import ExcelWriter, dataframe2excel
writer = ExcelWriter(theme_color="2639E9")
dataframe2excel(bins, writer, sheet_name="分箱分析", title="多头查询次数 · MOB6 > 30 分箱报告", percent_cols=["坏样本率", "样本占比"], condition_cols=["LIFT值"], color_cols=["坏样本率"], auto_width=True, figures=[str(out / "binning-plot.png"), str(out / "binning-trend.png")])
sheet = writer.get_sheet_by_name("分箱分析")
writer.set_freeze_panes(sheet, "D5")
writer.set_column_width(sheet, "B", 26)
writer.set_row_height(sheet, 2, 30)
writer.insert_value2sheet(sheet, "B1", "查看透视分析", bold=True, font_color="2639E9")
writer.insert_hyperlink2sheet(sheet, "B1", sheet="透视分析", target_space="B2")
detail = df[["GROUP", "授信月份", "loan_amt"]].fillna("")
raw_sheet = writer.get_sheet_by_name("数据")
writer.insert_df2sheet(raw_sheet, detail, "A1", index=False, auto_width=True)
writer.insert_pivot_table2sheet(worksheet=writer.get_sheet_by_name("透视分析"), pivot_anchor="B2", source_sheet=raw_sheet, source_range=f"A1:C{len(detail)+1}", rows="GROUP", columns="授信月份", values=[{"field": "loan_amt", "agg": "sum", "name": "额度合计", "number_format": "#,##0.00"}, {"field": "loan_amt", "agg": "sum", "show_as": "全局占比", "name": "金额占比", "number_format": "0.00%"}], name="客群授信月份额度透视表", theme_style=True, fill=True)
for worksheet in writer.workbook.worksheets:
    worksheet.sheet_view.showGridLines = False
    worksheet.sheet_view.showRowColHeaders = False
writer.save(str(out / "excel-showcase.xlsx"))

## Swap 与多规则报告

In [ ]:
from hscredit.report.swap_analysis import create_swap_dataset_from_rules, swap_analysis
from hscredit.report import swap_out_report
# 只在独立测试集评估策略；参考分数为 MOB6 > 30 的模型预测概率。
swap_reference = test.assign(模型风险概率=probability)
swap_data = create_swap_dataset_from_rules(swap_reference, original_rule=rule1, new_rule=rule2, score_col="模型风险概率", n_jobs=1)
swap = swap_analysis(swap_data, reference_df=swap_reference, score_col="模型风险概率", overdue=["MOB1", "MOB6"], dpds=[0, 30], n_jobs=1)
from hscredit.utils.overdue import overdue_label
show_table(swap.get_summary_report(target=overdue_label("MOB6", 30, ">")), "swap-summary")
for label in swap.targets:
    show_table(swap.get_summary_report(target=label), "swap-" + label)
strategy = swap_out_report(test, rules=[rule1, rule2], overdue=["MOB1", "MOB6"], dpds=[0, 30], features=["最近12个月多头查询次数"], date="授信时间", methods="quantile", save=str(out / "swap-out.xlsx"), n_jobs=1)
# 该入口的 date / group 互斥，月份与客群分别交付。
strategy_cohort = swap_out_report(test, rules=[rule1, rule2], overdue=["MOB1", "MOB6"], dpds=[0, 30], features=["最近12个月多头查询次数"], group="GROUP", methods="quantile", save=str(out / "swap-out-cohort.xlsx"), n_jobs=1)

## 自定义分箱目标

In [ ]:
from hscredit.core.binning import ORBinning, CustomObjectives
custom_binner = ORBinning(max_n_bins=4, objective=CustomObjectives.max_lift_iv(lift_weight=0.1), n_jobs=1)
custom_binner.fit(X_train[["最近12个月多头查询次数"]], y_train)
custom_bins = feature_bin_stats(test, feature="最近12个月多头查询次数", target="坏样本", binner=custom_binner, margins=True, return_cols=["分箱标签", "样本总数", "坏样本率", "LIFT值", "指标IV值"], n_jobs=1)
show_table(custom_bins, "binning-custom")

## 二叉与多叉树规则

In [ ]:
from hscredit.report.mining import ManualTreeExtractor, MultiwayTreeExtractor
# 仅训练集决定根节点和手工拆分；测试集只用于报告。
tree_train, tree_test = train.copy(), test.copy()
tree_train[features], tree_test[features] = X_num, X_test_num
tree_train["坏样本"], tree_test["坏样本"] = y_train, y_test
binary_tree = ManualTreeExtractor(target="坏样本", features=features, max_depth=2, min_samples_leaf=100, random_state=42, n_jobs=1)
binary_tree.fit(tree_train, features=features)
binary_node = 1
assert binary_node != 0
binary_root_before = (binary_tree._tree_info.feature[0], binary_tree._tree_info.threshold[0])
binary_tree.manual_split(tree_train, feature="贷款本月应还金额", threshold=20000, node=binary_node)
assert binary_root_before == (binary_tree._tree_info.feature[0], binary_tree._tree_info.threshold[0])
assert all(edit["节点编号"] != 0 for edit in binary_tree.edit_history_)
show_table(binary_tree.get_rule_table(tree_test), "tree-binary-rules")
show_plot("tree-binary", viz.plot_tree, binary_tree, backend="matplotlib")

multiway_tree = MultiwayTreeExtractor(target="坏样本", features=features, max_depth=2, max_n_bins=3, min_samples_leaf=100, n_jobs=1)
multiway_tree.fit(tree_train, features=features)
node_table = multiway_tree.get_node_table().set_index("节点编号")
multiway_node = int(node_table.loc[0, "子节点"][0])
assert multiway_node != 0
multiway_root_before = (node_table.loc[0, "分裂特征"], tuple(node_table.loc[0, "切分点"]))
multiway_tree.manual_split(node=multiway_node, feature="贷款本月应还金额", thresholds=[5000, 20000], grow=False)
node_table = multiway_tree.get_node_table().set_index("节点编号")
assert multiway_root_before == (node_table.loc[0, "分裂特征"], tuple(node_table.loc[0, "切分点"]))
assert node_table.loc[0, "人工分裂"] == "否"
show_table(multiway_tree.get_rule_table(tree_test), "tree-multiway-rules")
show_plot("tree-multiway", viz.plot_tree, multiway_tree, backend="matplotlib")

# 原生树报告同时覆盖训练 / 测试集、四个逾期标签、授信月份及客群。
tree_datasets = {"训练集": tree_train, "测试集": tree_test}
for name, tree in [("binary", binary_tree), ("multiway", multiway_tree)]:
    tree_reports = tree.report(
        datasets=tree_datasets, overdue=["MOB1", "MOB6"], dpds=[0, 30],
        overdue_operator=">", amount="loan_amt", date="授信时间", group="GROUP",
        methods=["quantile"], bin_params={"max_n_bins": 4}, leaf_only=False,
        save=str(out / f"{name}-tree-report.xlsx"),
    )
    assert set(tree_reports) == set(tree_datasets)
    for dataset, table in tree_reports.items():
        show_table(table, f"tree-{name}-{dataset}")


## 分类、回归、多标签报告

In [ ]:
from hscredit.report import auto_model_report
report_train, report_test = train.copy(), test.copy()
report_train[features] = X_num
report_test[features] = X_test_num
classification = auto_model_report(model, datasets={"训练集": report_train, "测试集": report_test}, features=features, target="坏样本", with_plots=False, n_jobs=1, verbose=False)
show_table(classification.get_metrics(), "report-classification")
multi_label = auto_model_report(model, datasets={"训练集": report_train, "测试集": report_test}, features=features, overdue=["MOB1", "MOB6"], dpds=[0, 30], overdue_operator=">", amount="loan_amt", date="授信时间", group="GROUP", long_format=True, excel_path=str(out / "auto-model-report.xlsx"), with_plots=True, n_jobs=1, verbose=False)
show_table(multi_label.summary(), "report-multi-label")
from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(n_estimators=40, max_depth=4, n_jobs=1, random_state=42)
regressor.fit(X_num, train["loan_amt"])
regression = auto_model_report(regressor, datasets={"训练集": report_train, "测试集": report_test}, features=features, target="loan_amt", excel_path=str(out / "regression-model-report.xlsx"), n_jobs=1, verbose=False)
show_table(regression.get_metrics(), "report-regression")
# 验证每一标签的测试样本分母和坏率，不能误展示 combined y。
for mob in ["MOB1", "MOB6"]:
    for dpd in [0, 30]:
        observed = test[mob].dropna()
        row = multi_label.summary().loc[f"{mob}@{dpd}"]
        assert row[("样本数", "测试集")] == len(observed)
        np.testing.assert_allclose(row[("坏样本率", "测试集")], observed.gt(dpd).mean())


## 自定义损失与推理导出

In [ ]:
from hscredit.core.models.losses import FocalLoss
risk_model = XGBoost(objective=FocalLoss(gamma=2.0), n_estimators=20, max_depth=3, n_jobs=1, random_state=42)
risk_model.fit(X_num, y_train)
# 保存评分卡，分别供 PMML 评分引擎和 Python 服务使用
with tempfile.TemporaryDirectory() as temporary:
    directory = Path(temporary)
    card.export_pmml(str(directory / "scorecard.pmml"))
    card.save_pickle(str(directory / "scorecard.pkl"))
    serving_card = ScoreCard.load_pickle(str(directory / "scorecard.pkl"))
    np.testing.assert_allclose(serving_card.predict(X_test), card.predict(X_test))


## 多标签特征分析报告

In [ ]:
from hscredit.report import auto_feature_analysis
feature_report = auto_feature_analysis(df, features=analysis_features, overdue=["MOB1", "MOB6"], dpds=[0, 30], overdue_operator=">", amount="loan_amt", date="授信时间", freq="M", corr=False, long_format=True, bin_params={"method": "quantile", "max_n_bins": 4}, excel_writer=str(out / "feature-analysis.xlsx"), output_dir=str(work / "feature-analysis"), margins=True, n_jobs=1, show_progress=False)

## 可选扩展

TabPFN 权重、数据库连接及报告完整批量运行示例见 README 对应折叠代码与各专题 Notebook。当前 Notebook 不连接外部数据库、不下载模型权重。